In [ ]:
from skipalignments import *
############### ENTER THE LOG PATHS HERE ###############
path_to_road_fines_log = '.../path/to/xes'
path_to_request_for_payment_log = '.../path/to/xes'
path_to_international_declarations_log = '.../path/to/xes'
inspected_log = Logs.ROAD_FINES

############### ENTER THE STOCHASTIC ESTIMATOR HERE ###############
ebi_method = EbiWeights.OCCURANCE

In [ ]:
%load_ext autoreload
%autoreload 2
import pm4py
import statistics
import random

In [ ]:
def update_pair_taus(tree:ProcessTree):
    if isinstance(tree, Tau):
        if tree.parent is not None and len(tree.parent.children) == 2:
            other = tree.parent.children[0]
            if other == tree:
                other = tree.parent.children[1]
            if isinstance(other, Activity):
                # set tau
                tree.name = "TAU_" + other.name
            else:
                tree.name = "TAU_" + other.id
        else:
            tree.name = "TAU_" + str(tree.get_distance_to_root()) + str(random.random())
        return
    elif not isinstance(tree, Activity):
        for c in tree.children:
            update_pair_taus(c)
        return
    else:
        return

In [ ]:
def check_names(tree:ProcessTree, names:List[str]):
    if isinstance(tree, Activity):
        assert tree.name in names
    elif isinstance(tree, Tau):
        pass
    else:
        for c in tree.children:
            check_names(c, names)

In [ ]:
path = None
if inspected_log == Logs.ROAD_FINES:
    path = path_to_road_fines_log
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    path = path_to_request_for_payment_log
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    path = path_to_international_declarations_log
log_rf = pm4py.read_xes(path)

In [ ]:
if inspected_log == Logs.ROAD_FINES:
    ## rf
    createfine = Activity(None, 'Create Fine', 100000)
    appealtojudge = Activity(None, 'Appeal to Judge', 100000)
    insertdateappealtoprefecture = Activity(None, 'Insert Date Appeal to Prefecture', 100000)
    receiveresultappealfromprefecture = Activity(None, 'Receive Result Appeal from Prefecture', 100000)
    notifyresultappealtooffender = Activity(None, 'Notify Result Appeal to Offender', 100000)
    sendappealtoprefecture = Activity(None, 'Send Appeal to Prefecture', 100000)
    payment = Activity(None, 'Payment', 100000)
    sendfine = Activity(None, 'Send Fine', 100000)
    insertfinenotification = Activity(None, 'Insert Fine Notification', 100000)
    addpenality = Activity(None, 'Add penalty', 100000)
    sendforcreditcollection = Activity(None, 'Send for Credit Collection', 100000)

    atjtau = Tau(None, 'TAU_Appeal to Judge', 0)
    nratotau = Tau(None, 'TAU_Notify Result Appeal to Offender', 0)
    ptau = Tau(None, 'TAU_Payment', 0)
    sfcctau = Tau(None, 'Send for Credit Collection', 0)

    atjchoice = Xor(None, [atjtau, appealtojudge])
    atjtau.set_parent(atjchoice)
    appealtojudge.set_parent(atjchoice)
    nratochoice = Xor(None, [nratotau, notifyresultappealtooffender])
    nratotau.set_parent(nratochoice)
    notifyresultappealtooffender.set_parent(nratochoice)
    pchoice = Xor(None, [ptau, payment])
    ptau.set_parent(pchoice)
    payment.set_parent(pchoice)
    sfccchoice = Xor(None, [sfcctau, sendforcreditcollection])
    sfcctau.set_parent(sfccchoice)
    sendforcreditcollection.set_parent(sfccchoice)

    idatpnratoseq = Sequence(None, [insertdateappealtoprefecture, nratochoice])
    insertdateappealtoprefecture.set_parent(idatpnratoseq)
    nratochoice.set_parent(idatpnratoseq)
    ifnapseq = Sequence(None, [insertfinenotification, addpenality])
    insertfinenotification.set_parent(ifnapseq)
    addpenality.set_parent(ifnapseq)

    tauand1 = Tau(None, 'TAU_AND1', 0)
    tauseq1 = Tau(None, 'TAU_SEQ1', 0)
    tauseq2 = Tau(None, 'TAU_SEQ2', 0)

    ifnapchoice = Xor(None, [tauseq1, ifnapseq])
    tauseq1.set_parent(ifnapchoice)
    ifnapseq.set_parent(ifnapchoice)
    sfifnapseq = Sequence(None, [sendfine, ifnapchoice])
    sendfine.set_parent(sfifnapseq)
    ifnapchoice.set_parent(sfifnapseq)
    sfifnapchoice = Xor(None, [tauseq2, sfifnapseq])
    tauseq2.set_parent(sfifnapchoice)
    sfifnapseq.set_parent(sfifnapchoice)

    and1 = And(None, [atjchoice, receiveresultappealfromprefecture, idatpnratoseq, sendappealtoprefecture])
    atjchoice.set_parent(and1)
    receiveresultappealfromprefecture.set_parent(and1)
    idatpnratoseq.set_parent(and1)
    sendappealtoprefecture.set_parent(and1)
    and2 = And(None, [pchoice, sfifnapchoice])
    pchoice.set_parent(and2)
    sfifnapchoice.set_parent(and2)

    choice = Xor(None, [tauand1, and1])
    tauand1.set_parent(choice)
    and1.set_parent(choice)

    tree_rf = Sequence(None, [createfine, choice, and2, sfccchoice])
    createfine.set_parent(tree_rf)
    choice.set_parent(tree_rf)
    and2.set_parent(tree_rf)
    sfccchoice.set_parent(tree_rf)

    check_names(tree_rf, list(log_rf['concept:name'].unique()))
    tree_rf
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    ## payreq
    submittedbyemployee = Activity(None, 'Request For Payment SUBMITTED by EMPLOYEE', 100000)
    submittedbyemployee2 = Activity(None, 'Request For Payment SUBMITTED by EMPLOYEE', 100000)
    rejectedbyadministration = Activity(None, 'Request For Payment REJECTED by ADMINISTRATION', 100000)
    rejectedbyadministration2 = Activity(None, 'Request For Payment REJECTED by ADMINISTRATION', 100000)
    approvedbyadministration = Activity(None, 'Request For Payment APPROVED by ADMINISTRATION', 100000)
    approvedbyadministration2 = Activity(None, 'Request For Payment APPROVED by ADMINISTRATION', 100000)
    rejectedbysupervisor = Activity(None, 'Request For Payment REJECTED by SUPERVISOR', 100000)
    submittedbyemployee3 = Activity(None, 'Request For Payment SUBMITTED by EMPLOYEE', 100000)
    approvedbybudgetowner = Activity(None, 'Request For Payment APPROVED by BUDGET OWNER', 100000)
    rejectedbyemployee = Activity(None, 'Request For Payment REJECTED by EMPLOYEE', 100000)
    finalapprovedbysupervisor = Activity(None, 'Request For Payment FINAL_APPROVED by SUPERVISOR', 100000)
    finalapprovedbydirector = Activity(None, 'Request For Payment FINAL_APPROVED by DIRECTOR', 100000)
    approvedbysupervisor = Activity(None, 'Request For Payment APPROVED by SUPERVISOR', 100000)
    requestpayment = Activity(None, 'Request Payment', 100000)
    finalapprovedbybudgetowner = Activity(None, 'Request For Payment FINAL_APPROVED by BUDGET OWNER', 100000)
    paymenthandled = Activity(None, 'Payment Handled', 100000)
    savedbyemployee = Activity(None, 'Request For Payment SAVED by EMPLOYEE', 100000)
    forapprovalbysupervisor = Activity(None, 'Request For Payment FOR_APPROVAL by SUPERVISOR', 100000)

    tau1 = Tau(None, 'TAU_LOOPS', 0)
    tau2 = Tau(None, 'TAU_AND', 0)
    tau3 = Tau(None, 'TAU_FINAL_APPROVED by BUDGET OWNER', 0)
    tau4 = Tau(None, 'TAU_XOR', 0)

    aarachoice = Xor(None, [approvedbyadministration, rejectedbyadministration2])
    approvedbyadministration.set_parent(aarachoice)
    rejectedbyadministration2.set_parent(aarachoice)

    seaaraseq = Sequence(None, [submittedbyemployee, aarachoice])
    submittedbyemployee.set_parent(seaaraseq)
    aarachoice.set_parent(seaaraseq)

    seraloop = Loop(None, [submittedbyemployee2, rejectedbyadministration])
    submittedbyemployee2.set_parent(seraloop)
    rejectedbyadministration.set_parent(seraloop)

    seraaaseq = Sequence(None, [seraloop, approvedbyadministration2])
    seraloop.set_parent(seraaaseq)
    approvedbyadministration2.set_parent(seraaaseq)

    seraaarsloop = Loop(None, [seraaaseq, rejectedbysupervisor])
    seraaaseq.set_parent(seraaarsloop)
    rejectedbysupervisor.set_parent(seraaarsloop)

    seraaarsabseq = Sequence(None, [seraaarsloop, approvedbybudgetowner])
    seraaarsloop.set_parent(seraaarsabseq)
    approvedbybudgetowner.set_parent(seraaarsabseq)

    xor1 = Xor(None, [seaaraseq, seraaarsabseq, submittedbyemployee3])
    seaaraseq.set_parent(xor1)
    seraaarsabseq.set_parent(xor1)
    submittedbyemployee3.set_parent(xor1)

    loop = Loop(None, [xor1, rejectedbyemployee])
    xor1.set_parent(loop)
    rejectedbyemployee.set_parent(loop)

    xor = Xor(None, [tau1, loop])
    tau1.set_parent(xor)
    loop.set_parent(xor)


    and1 = And(None, [finalapprovedbydirector, approvedbysupervisor])
    finalapprovedbydirector.set_parent(and1)
    approvedbysupervisor.set_parent(and1)

    block3 = Xor(None, [tau2, and1])
    tau2.set_parent(block3)
    and1.set_parent(block3)

    block5 = Xor(None, [tau3, finalapprovedbybudgetowner])
    tau3.set_parent(block5)
    finalapprovedbybudgetowner.set_parent(block5)

    block7 = Xor(None, [tau4, savedbyemployee, forapprovalbysupervisor])
    tau4.set_parent(block7)
    savedbyemployee.set_parent(block7)
    forapprovalbysupervisor.set_parent(block7)

    tree_rf = Sequence(None, [xor, finalapprovedbysupervisor, block3, requestpayment, block5, paymenthandled, block7])
    xor.set_parent(tree_rf)
    finalapprovedbysupervisor.set_parent(tree_rf)
    block3.set_parent(tree_rf)
    requestpayment.set_parent(tree_rf)
    block5.set_parent(tree_rf)
    paymenthandled.set_parent(tree_rf)
    block7.set_parent(tree_rf)

    check_names(tree_rf, list(log_rf['concept:name'].unique()))
    tree_rf
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    ## international declarations
    permitsbe = Activity(None, 'Permit SUBMITTED by EMPLOYEE', 100000)
    permitrba = Activity(None, 'Permit REJECTED by ADMINISTRATION', 100000)
    declarationrbe = Activity(None, 'Declaration REJECTED by EMPLOYEE', 100000)
    permitaba = Activity(None, 'Permit APPROVED by ADMINISTRATION', 100000)
    permitabbo = Activity(None, 'Permit APPROVED by BUDGET OWNER', 100000)
    permitfabs = Activity(None, 'Permit FINAL_APPROVED by SUPERVISOR', 100000)
    startt = Activity(None, 'Start trip', 100000)
    endt = Activity(None, 'End trip', 100000)
    declarationsbe = Activity(None, 'Declaration SUBMITTED by EMPLOYEE', 100000)
    declarationaba = Activity(None, 'Declaration APPROVED by ADMINISTRATION', 100000)
    declarationabbo = Activity(None, 'Declaration APPROVED by BUDGET OWNER', 100000)
    declarationfabs = Activity(None, 'Declaration FINAL_APPROVED by SUPERVISOR', 100000)
    requestp = Activity(None, 'Request Payment', 100000)
    paymenth = Activity(None, 'Payment Handled', 100000)
    tau = Tau(None, 'TAU_Declaration APPROVED by BUDGET OWNER', 0)

    loop = Loop(None, [permitsbe, permitrba])
    permitsbe.set_parent(loop)
    permitrba.set_parent(loop)

    choice1 = Xor(None, [declarationrbe, loop])
    declarationrbe.set_parent(choice1)
    loop.set_parent(choice1)

    choice2 = Xor(None, [tau, declarationabbo])
    tau.set_parent(choice2)
    declarationabbo.set_parent(choice2)

    tree_rf = Sequence(None, [choice1, permitaba, permitabbo, permitfabs, startt, endt, declarationsbe, declarationaba, choice2, declarationfabs, requestp, paymenth])
    choice1.set_parent(tree_rf)
    permitaba.set_parent(tree_rf)
    permitabbo.set_parent(tree_rf)
    permitfabs.set_parent(tree_rf)
    startt.set_parent(tree_rf)
    endt.set_parent(tree_rf)
    declarationsbe.set_parent(tree_rf)
    declarationaba.set_parent(tree_rf)
    choice2.set_parent(tree_rf)
    declarationfabs.set_parent(tree_rf)
    requestp.set_parent(tree_rf)
    paymenth.set_parent(tree_rf)

    check_names(tree_rf, list(log_rf['concept:name'].unique()))
    tree_rf

In [ ]:
process_tree_rf = tree_rf.to_pm4py()
pm4py.view_process_tree(process_tree_rf, format='png')
tree_rf = ProcessTree.from_pm4py(process_tree_rf, 100000, 0, 0)

In [ ]:
update_pair_taus(tree_rf)

In [ ]:
tree_rf

In [ ]:
output_path = None
if inspected_log == Logs.ROAD_FINES:
    output_path = "./results/indulpet/rf"
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    output_path = "./results/indulpet/payment"
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    output_path = "./results/indulpet/declarations"

if ebi_method == EbiWeights.OCCURANCE:
    output_path += "/occurance"
elif ebi_method == EbiWeights.UNIFORM:
    output_path += "/uniform"

In [ ]:
derivation = DerivationPipeline(tree_rf, log_rf, pn_log=log_rf, pn_method=ebi_method, sagn_timeout=600)

In [ ]:
smodel_path = "smodel.slpn"
if inspected_log == Logs.REQUEST_FOR_PAYMENT:
    smodel_path = "./indulpet_results/payment/occurance/smodel.slpn"
derivation.compute(output_path, slpn_path=smodel_path)

In [ ]:
print(derivation.print_blinded())

In [ ]:
derivation.stats()